In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [2]:
train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [3]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [4]:
test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [5]:
print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [6]:
TARGET="class"
ID_COL="id"

train_ids=train[ID_COL]
test_ids=test[ID_COL]

X=train.drop(columns=[TARGET, ID_COL])
y=train[TARGET]
X_test=test.drop(columns=[ID_COL])

In [7]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [ ]:
def feature_set_3(df):

    df = df.copy()
    # feature set 1
    df["u_g"]=df["u"] - df["g"]
    df["g_r"]=df["g"] - df["r"]
    df["r_i"]=df["r"] - df["i"]
    df["i_z"]=df["i"] - df["z"]
    df["u_r"]=df["u"] - df["r"]
    df["u_i"]=df["u"] - df["i"]
    df["u_z"]=df["u"] - df["z"]
    df["g_i"]=df["g"] - df["i"]
    df["g_z"]=df["g"] - df["z"]
    df["r_z"]=df["r"] - df["z"]

    # feature set 2
    eps=1e-6
    df["log_redshift"]=np.log1p(df["redshift"])
    df["sqrt_redshift"]=np.sqrt(np.clip(df["redshift"], 0, None))
    df["redshift_sq"]=(df["redshift"]**2)
    df["redshift_u"]=(df["redshift"] * df["u"])
    df["redshift_g"]=(df["redshift"] * df["g"])
    df["redshift_r"]=(df["redshift"] * df["r"])
    df["redshift_i"]=(df["redshift"] * df["i"])
    df["redshift_z"]=(df["redshift"] * df["z"])
    df["redshift_u_g"]=(df["redshift"] * df["u_g"])
    df["redshift_g_r"]=(df["redshift"] * df["g_r"])
    df["redshift_r_i"]=(df["redshift"] * df["r_i"])
    df["redshift_i_z"]=(df["redshift"] * df["i_z"])
    df["redshift_div_u"]=(df["redshift"] / (np.abs(df["u"]) + eps))
    df["redshift_div_g"]=(df["redshift"] / (np.abs(df["g"]) + eps))
    df["redshift_div_r"]=(df["redshift"] / (np.abs(df["r"]) + eps))
    df["redshift_div_i"]=(df["redshift"] / (np.abs(df["i"]) + eps))
    df["redshift_div_z"]=(df["redshift"] / (np.abs(df["z"]) + eps))
    
    # FEATURE SET 3
    # Spectral curvature
    df["curv_1"]=(df["u_g"] - df["g_r"])
    df["curv_2"]=(df["g_r"] - df["r_i"])
    df["curv_3"]=(df["r_i"] - df["i_z"])

    # Spectrum spread
    mag_cols=["u","g","r","i","z"]

    df["mag_std"]=(df[mag_cols].std(axis=1))
    df["mag_range"]=(df[mag_cols].max(axis=1))-(df[mag_cols].min(axis=1))

    df["alpha_sin"]=np.sin(np.radians(df["alpha"]))
    df["alpha_cos"]=np.cos(np.radians(df["alpha"]))

    df["delta_sin"]=np.sin(np.radians(df["delta"]))
    df["delta_cos"]=np.cos(np.radians(df["delta"]))
    
    return df

In [ ]:
X_fs3=feature_set_3(X)
X_test_fs3=feature_set_3(X_test)

print(X_fs3.shape)
print(X_test_fs3.shape)
print(X_fs3.columns.to_list())
print(X_test_fs3.columns.to_list())

(577347, 46)
(247435, 46)
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_z', 'curv_1', 'curv_2', 'curv_3', 'mag_std', 'mag_range', 'alpha_sin', 'alpha_cos', 'delta_sin', 'delta_cos']
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_

In [14]:
X_fs3.to_csv("X_fs3.csv", index=False)
X_test_fs3.to_csv("X_test_fs3.csv", index=False)

In [ ]:
cat_cols = ["spectral_type","galaxy_population"]

ohe=ColumnTransformer(transformers=[("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)],
    remainder="passthrough")

X_ohe = ohe.fit_transform(X_fs3)
X_test_ohe = ohe.transform(X_test_fs3)

print(X_ohe.shape)
print(X_test_ohe.shape)

(577347, 50)
(247435, 50)


In [ ]:
X_mi=pd.get_dummies(X_fs3,drop_first=True)

mi_scores=mutual_info_classif(X_mi,y_encoded,random_state=42)

mi_df=pd.DataFrame({"Feature": X_mi.columns,"MI": mi_scores})
mi_df.sort_values("MI",ascending=False).head(60)

,Feature,MI
30,redshift_div_u,0.525462
31,redshift_div_g,0.523478
32,redshift_div_r,0.516961
7,redshift,0.514988
18,log_redshift,0.514986
19,sqrt_redshift,0.514983
20,redshift_sq,0.514618
33,redshift_div_i,0.512376
25,redshift_z,0.511831
34,redshift_div_z,0.509393


In [ ]:
et=ExtraTreesClassifier(n_estimators=500,random_state=42,n_jobs=-1)
et.fit(X_mi,y)
et_df = pd.DataFrame({
    "Feature": X_mi.columns,
    "Importance": et.feature_importances_})
et_df.sort_values("Importance",ascending=False).head(60)

,Feature,Importance
45,qso_vs_gal,0.074134
44,star_vs_gal,0.063864
47,spectral_type_M,0.060152
18,log_redshift,0.043344
43,star_vs_qso,0.039250
49,galaxy_population_Red_Sequence,0.035305
19,sqrt_redshift,0.035049
25,redshift_z,0.031175
7,redshift,0.030076
31,redshift_div_g,0.029538


In [ ]:
lgbm_imp_model=LGBMClassifier(random_state=42,verbose=-1)
lgbm_imp_model.fit(X_ohe,y)

imp_df = pd.DataFrame({
    "Feature": ohe.get_feature_names_out(),
    "Importance": lgbm_imp_model.feature_importances_
})

imp_df.sort_values("Importance",ascending=False).head(50)

,Feature,Importance
7,remainder__delta,1530
6,remainder__alpha,1513
12,remainder__z,388
22,remainder__g_z,354
9,remainder__g,338
8,remainder__u,299
10,remainder__r,296
11,remainder__i,282
14,remainder__u_g,271
36,remainder__redshift_div_u,250


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)

In [ ]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=900,
    max_depth=8,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
scores = cross_val_score(xgb,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy")
print("Mean CV:",scores.mean())
print("Std CV :",scores.std())

Mean CV: 0.9557523243738679
Std CV : 0.000748938596907418


In [36]:
xgb.fit(X_ohe, y_encoded)
test_predictions = xgb.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

In [ ]:
submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission_fset32.csv", index=False)

print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY


In [ ]:
lgbm2 = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.01,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgbm_scores2=cross_val_score(lgbm2,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)
print("Mean CV :", lgbm_scores2.mean())
print("Std CV  :", lgbm_scores2.std())

Mean CV : 0.9577257290605334
Std CV  : 0.000721466891281101


In [ ]:
lgbm2.fit(X_ohe, y_encoded)
test_predictions = lgbm2.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

submission_df=pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission__lgbm5.csv", index=False)
print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created and saved as 'submission.csv'!
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
